# 03 · MLP V2 (불균형 대응 6변형)

> processed_v2(42피처)에서 MLP를 6가지로 비교한다: plain BCE / weighted / focal / focal+dropout / focal+batchnorm / SMOTE.
> 비교 기준: 기존 최고 MLP(MLP_3_focal) 0.193 / 트리 최고 XGB_v2_tuned 0.296.
> 현실 목표: 0.20~0.23 (트리 0.296은 못 넘을 가능성 큼 - 그래도 정직하게 기록).
> 기록: results_v2.csv.

## 0. 환경 설정 + 데이터

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR    = PROJECT_ROOT / "processed_v2"
NB_DIR     = PROJECT_ROOT / "notebooks_v2"
RESULTS_V2 = NB_DIR / "results_v2.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]
X_train = train_df[feature_cols].values.astype("float32")
y_train = train_df[TARGET].values.astype("float32")
X_val   = val_df[feature_cols].values.astype("float32")
y_val   = val_df[TARGET].values.astype("float32")
X_val_t = torch.from_numpy(X_val).to(device)
print("device:", device, " 피처:", len(feature_cols))

device: cpu  피처: 42


## 1. 모델 / 손실 / 학습 함수
- make_mlp: 입력 -> 64 -> 32 -> 1. dropout/batchnorm은 켜고 끌 수 있음.
- FocalLoss: 어려운 소수 샘플에 집중.
- train_model: 손실 방식(bce/weighted/focal)만 바꿔 학습하고 val 확률 반환.

In [2]:
def make_mlp(input_dim, use_dropout=False, use_batchnorm=False):
    layers = []
    sizes = [input_dim, 64, 32]
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(sizes[i + 1]))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(0.3))
    layers.append(nn.Linear(32, 1))
    return nn.Sequential(*layers)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

def make_loader(X, y, batch_size=4096, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

def train_model(model, loader, loss_kind="bce", epochs=8, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if loss_kind == "weighted":
        pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                                  dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    elif loss_kind == "focal":
        loss_fn = FocalLoss()
    else:
        loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb).squeeze(1), yb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X_val_t).squeeze(1)).cpu().numpy()

## 2. 실험 1~5 (손실/정규화 변형)
같은 train_loader로 손실 방식과 정규화만 바꿔 비교. 결과는 results_v2.csv에 기록.

In [3]:
train_loader = make_loader(X_train, y_train)
runs = {}

def run_experiment(name, model, loss_kind, loader=None):
    val_prob = train_model(model, loader if loader is not None else train_loader, loss_kind=loss_kind)
    m = compute_metrics(y_val, val_prob)
    log_result(name, m, path=str(RESULTS_V2))
    runs[name] = (val_prob, m)
    print("{:26s} PR_AUC {:.3f}  Recall {:.3f}  F1 {:.3f}".format(name, m["PR_AUC"], m["Recall"], m["F1"]))

input_dim = len(feature_cols)
set_seed(42); run_experiment("MLP_v2_plain_BCE",     make_mlp(input_dim),              "bce")
set_seed(42); run_experiment("MLP_v2_weighted",      make_mlp(input_dim),              "weighted")
set_seed(42); run_experiment("MLP_v2_focal",         make_mlp(input_dim),              "focal")
set_seed(42); run_experiment("MLP_v2_focal_dropout", make_mlp(input_dim, True, False), "focal")
set_seed(42); run_experiment("MLP_v2_focal_batchnorm", make_mlp(input_dim, False, True), "focal")

MLP_v2_plain_BCE           PR_AUC 0.198  Recall 0.000  F1 0.000


MLP_v2_weighted            PR_AUC 0.177  Recall 0.859  F1 0.097


MLP_v2_focal               PR_AUC 0.195  Recall 0.165  F1 0.222


MLP_v2_focal_dropout       PR_AUC 0.185  Recall 0.107  F1 0.166


MLP_v2_focal_batchnorm     PR_AUC 0.195  Recall 0.163  F1 0.219


## 3. 실험 6 (SMOTE)
train에만 SMOTE로 소수 클래스를 10%까지 합성한 뒤 plain BCE로 학습.

In [4]:
sm = SMOTE(sampling_strategy=0.1, k_neighbors=5, random_state=42)
X_sm, y_sm = sm.fit_resample(X_train, y_train.astype("int"))
X_sm = X_sm.astype("float32"); y_sm = y_sm.astype("float32")
print("SMOTE 후:", np.bincount(y_sm.astype(int)))

sm_loader = make_loader(X_sm, y_sm)
set_seed(42); run_experiment("MLP_v2_SMOTE", make_mlp(input_dim), "bce", loader=sm_loader)

SMOTE 후: [1341254  134125]


MLP_v2_SMOTE               PR_AUC 0.192  Recall 0.544  F1 0.239


## 4. 결과 비교 + 최고 MLP 저장
PR_AUC로 최고 MLP를 고르고, 그 확률을 저장(05 앙상블/06 threshold용).

In [5]:
best_name, best_pr = None, -1
for name in runs:
    pr = runs[name][1]["PR_AUC"]
    if pr > best_pr:
        best_pr, best_name = pr, name
np.save(NB_DIR / "mlp_v2_best_val_prob.npy", runs[best_name][0])
print("최고 MLP_v2:", best_name, "PR_AUC", round(best_pr, 4))
print("기존 MLP_3_focal 0.193 / 트리 XGB_v2_tuned 0.296\n")

res = pd.read_csv(RESULTS_V2).drop_duplicates(subset="model", keep="last").sort_values("PR_AUC", ascending=False)
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

최고 MLP_v2: MLP_v2_plain_BCE PR_AUC 0.1982
기존 MLP_3_focal 0.193 / 트리 XGB_v2_tuned 0.296



,model,PR_AUC,ROC_AUC,Recall,Precision,F1
1,XGB_v2_tuned,0.2959,0.9687,0.8699,0.0861,0.1566
3,LGBM_v2_tuned,0.2636,0.9666,0.8907,0.0734,0.1356
0,XGB_v2_base,0.2418,0.9616,0.8911,0.0600,0.1124
2,LGBM_v2_base,0.2246,0.9568,0.8831,0.0551,0.1037
4,MLP_v2_plain_BCE,0.1982,0.9409,0.0000,0.0000,0.0000
8,MLP_v2_focal_batchnorm,0.1953,0.9476,0.1633,0.3306,0.2187
6,MLP_v2_focal,0.1952,0.9443,0.1651,0.3391,0.2221
9,MLP_v2_SMOTE,0.1924,0.9469,0.5440,0.1531,0.2389
7,MLP_v2_focal_dropout,0.1850,0.9427,0.1067,0.3691,0.1655
5,MLP_v2_weighted,0.1766,0.9464,0.8588,0.0515,0.0972


---
### 결론 (실행 후 해석)
- MLP_v2 최고 vs 기존 MLP 0.193: v2 피처가 MLP를 얼마나 올렸나.
- MLP_v2 최고 vs XGB_v2_tuned 0.296: 못 넘으면 정직하게 기록(정형=트리 강세 재확인).
- threshold 0.5의 F1은 참고용. 진짜 운영점은 06_Threshold_Optimization_V2에서.
- 어느 불균형 기법이 PR_AUC에 가장 좋았는지(보통 focal)도 ablation 소재.